In [0]:
parameters:
  - name: spn
    type: string
  - name: DATABRICKS_HOST
    type: string
  - name: TARGET_ENV
    type: string
  - name: TARGET_VOLUME_PATH
    type: string
  - name: VOLUME_FULL_NAME
    type: string
  - name: CATALOG_NAME
    type: string
  - name: SCHEMA_NAME
    type: string

steps:
- task: AzureCLI@2
  displayName: 'Upload Wheels to Unity Catalog Volume (Smart Skip - Databricks CLI)'
  inputs:
    azureSubscription: '${{ parameters.spn }}'
    addSpnToEnvironment: true
    scriptType: bash
    scriptLocation: inlineScript
    inlineScript: |
      set -e

      VOLUME_PATH="${{ parameters.TARGET_VOLUME_PATH }}"
      WHEELS_DIR="$(System.ArtifactsDirectory)/wheels"
      CATALOG="${{ parameters.CATALOG_NAME }}"
      SCHEMA="${{ parameters.SCHEMA_NAME }}"
      VOLUME_FULL_NAME="${{ parameters.VOLUME_FULL_NAME }}"

      # Normalize volume path - remove trailing slash
      VOLUME_PATH=$(echo "$VOLUME_PATH" | sed 's|/$||')

      echo "=========================================="
      echo "Upload Wheels to Unity Catalog Volume"
      echo "=========================================="
      echo "Environment:   ${{ parameters.TARGET_ENV }}"
      echo "Volume Path:   $VOLUME_PATH"
      echo "=========================================="
      echo ""

      # Acquire Azure AD token for Databricks and configure CLI
      echo "Configuring Databricks CLI..."
      export DATABRICKS_HOST="${{ parameters.DATABRICKS_HOST }}"
      export DATABRICKS_AAD_TOKEN=$(az account get-access-token \
        --resource 2ff814a6-3304-4ab8-85cb-cd0e6f879c1d \
        --query accessToken -o tsv)

      if [ -z "$DATABRICKS_AAD_TOKEN" ]; then
        echo "ERROR: Failed to get Databricks token"
        exit 1
      fi
      echo "Token acquired"
      echo ""

      # Verify wheels directory
      if [ ! -d "$WHEELS_DIR" ]; then
        echo "ERROR: Wheels directory not found"
        exit 1
      fi

      WHEEL_COUNT=$(ls -1 "$WHEELS_DIR"/*.whl 2>/dev/null | wc -l)
      if [ "$WHEEL_COUNT" -eq 0 ]; then
        echo "WARNING: No wheel files found"
        exit 0
      fi
      echo "Found $WHEEL_COUNT wheel file(s)"
      echo ""

      # List existing files in UC volume via Databricks CLI
      # UC volumes: use dbfs:/Volumes/ prefix with databricks fs ls
      echo "Analyzing existing files in volume..."
      DBFS_VOLUME_PATH="dbfs:${VOLUME_PATH}"
      echo "Listing: $DBFS_VOLUME_PATH"

      TEMP_FILE="/tmp/existing_$$.txt"
      > "$TEMP_FILE"

      # List files and extract name|size pairs into temp file
      # Redirect stderr to suppress errors if volume is empty; || true prevents exit
      databricks fs ls -l "$DBFS_VOLUME_PATH" 2>/dev/null > /tmp/ls_output_$$.txt || true

      grep "\.whl" /tmp/ls_output_$$.txt | while IFS= read -r line; do
        FNAME=$(echo "$line" | awk '{print $NF}' | xargs basename 2>/dev/null)
        FSIZE=$(echo "$line" | awk '{for(i=1;i<=NF;i++) if($i~/^[0-9]+$/) {print $i; exit}}')
        if [ -n "$FNAME" ] && [ -n "$FSIZE" ]; then
          echo "${FNAME}|${FSIZE}"
        fi
      done > "$TEMP_FILE" || true

      rm -f /tmp/ls_output_$$.txt

      EXISTING_COUNT=$(wc -l < "$TEMP_FILE" 2>/dev/null | tr -d ' ')
      echo "Found $EXISTING_COUNT existing file(s) in volume"
      echo ""

      # Upload with smart skip
      UPLOAD_COUNT=0
      SKIP_COUNT=0
      OVERWRITE_COUNT=0
      FAIL_COUNT=0
      FILE_NUM=0

      for file in "$WHEELS_DIR"/*.whl; do
        FILE_NUM=$((FILE_NUM + 1))
        FILENAME=$(basename "$file")

        # Get local file size
        if [ "$(uname)" = "Darwin" ]; then
          LOCAL_SIZE=$(stat -f%z "$file" 2>/dev/null)
        else
          LOCAL_SIZE=$(stat -c%s "$file" 2>/dev/null)
        fi

        # Get remote file size
        REMOTE_SIZE=$(grep "^${FILENAME}|" "$TEMP_FILE" 2>/dev/null | cut -d'|' -f2)

        # Decide action
        if [ -n "$REMOTE_SIZE" ] && [ "$LOCAL_SIZE" = "$REMOTE_SIZE" ]; then
          echo "[$FILE_NUM/$WHEEL_COUNT] $FILENAME - SKIP (identical, $LOCAL_SIZE bytes)"
          SKIP_COUNT=$((SKIP_COUNT + 1))
          continue
        elif [ -n "$REMOTE_SIZE" ]; then
          echo "[$FILE_NUM/$WHEEL_COUNT] $FILENAME - OVERWRITE (local: $LOCAL_SIZE bytes, remote: $REMOTE_SIZE bytes)"
          ACTION="overwrite"
        else
          echo "[$FILE_NUM/$WHEEL_COUNT] $FILENAME - NEW FILE ($LOCAL_SIZE bytes)"
          ACTION="new"
        fi

        TARGET_PATH="${DBFS_VOLUME_PATH}/${FILENAME}"
        echo "  Source: $file"
        echo "  Target: $TARGET_PATH"

        set +e
        UPLOAD_OUTPUT=$(databricks fs cp --overwrite "$file" "$TARGET_PATH" 2>&1)
        UPLOAD_EXIT=$?
        set -e

        if [ $UPLOAD_EXIT -eq 0 ]; then
          echo "  Result: SUCCESS"
          UPLOAD_COUNT=$((UPLOAD_COUNT + 1))
          [ "$ACTION" = "overwrite" ] && OVERWRITE_COUNT=$((OVERWRITE_COUNT + 1))
        else
          echo "  Result: FAILED (exit code: $UPLOAD_EXIT)"
          echo "  $UPLOAD_OUTPUT"
          FAIL_COUNT=$((FAIL_COUNT + 1))
        fi
        echo ""
      done

      rm -f "$TEMP_FILE"

      echo ""
      echo "=========================================="
      echo "Upload Summary"
      echo "=========================================="
      echo "Total files:         $WHEEL_COUNT"
      echo "Uploaded:            $UPLOAD_COUNT"
      echo "  - New files:       $((UPLOAD_COUNT - OVERWRITE_COUNT))"
      echo "  - Overwritten:     $OVERWRITE_COUNT"
      echo "Skipped (identical): $SKIP_COUNT"
      echo "Failed:              $FAIL_COUNT"
      echo "=========================================="
      echo ""

      if [ $FAIL_COUNT -gt 0 ]; then
        echo "ERROR: $FAIL_COUNT file(s) failed to upload!"
        exit 1
      fi

      if [ $SKIP_COUNT -eq $WHEEL_COUNT ]; then
        echo "All files already up-to-date - no upload needed!"
        echo "Time saved: ~90%"
      elif [ $SKIP_COUNT -gt 0 ]; then
        PERCENT=$((SKIP_COUNT * 100 / WHEEL_COUNT))
        echo "Upload completed with $SKIP_COUNT file(s) skipped!"
        echo "Time saved: ~${PERCENT}%"
      else
        echo "All files uploaded successfully!"
      fi

  env:
    DATABRICKS_HOST: ${{ parameters.DATABRICKS_HOST }}
    TARGET_ENV: ${{ parameters.TARGET_ENV }}
    DATABRICKS_AAD_TOKEN: $(DATABRICKS_AAD_TOKEN)


In [0]:
parameters:
  - name: spn
    type: string
  - name: DATABRICKS_HOST
    type: string
  - name: TARGET_ENV
    type: string
  - name: TARGET_VOLUME_PATH
    type: string
  - name: VOLUME_FULL_NAME
    type: string
  - name: CATALOG_NAME
    type: string
  - name: SCHEMA_NAME
    type: string

steps:
- task: AzureCLI@2
  displayName: 'Upload Wheels to Unity Catalog Volume (Smart Skip - CLI)'
  inputs:
    azureSubscription: '${{ parameters.spn }}'
    addSpnToEnvironment: true
    scriptType: bash
    scriptLocation: inlineScript
    inlineScript: |
      set -e

      VOLUME_PATH="${{ parameters.TARGET_VOLUME_PATH }}"
      WHEELS_DIR="$(System.ArtifactsDirectory)/wheels"
      CATALOG="${{ parameters.CATALOG_NAME }}"
      SCHEMA="${{ parameters.SCHEMA_NAME }}"
      VOLUME_FULL_NAME="${{ parameters.VOLUME_FULL_NAME }}"

      # Normalize volume path - remove trailing slash for consistency
      VOLUME_PATH=$(echo "$VOLUME_PATH" | sed 's|/$||')

      echo "=========================================="
      echo "Upload Wheels to Unity Catalog Volume"
      echo "=========================================="
      echo "Environment:   ${{ parameters.TARGET_ENV }}"
      echo "Volume Path:   $VOLUME_PATH"
      echo "=========================================="
      echo ""

      # Configure Databricks CLI with Azure token
      echo "Configuring Databricks CLI..."
      export DATABRICKS_HOST="${{ parameters.DATABRICKS_HOST }}"
      export DATABRICKS_TOKEN=$(az account get-access-token --resource 2ff814a6-3304-4ab8-85cb-cd0e6f879c1d --query accessToken -o tsv)

      if [ -z "$DATABRICKS_TOKEN" ]; then
        echo "ERROR: Failed to get Databricks token"
        exit 1
      fi

      echo "Token acquired successfully"
      echo ""

      # Verify wheels directory
      if [ ! -d "$WHEELS_DIR" ]; then
        echo "ERROR: Wheels directory not found"
        exit 1
      fi

      WHEEL_COUNT=$(ls -1 "$WHEELS_DIR"/*.whl 2>/dev/null | wc -l)

      if [ "$WHEEL_COUNT" -eq 0 ]; then
        echo "WARNING: No wheel files found"
        exit 0
      fi

      echo "Found $WHEEL_COUNT wheel file(s)"
      echo ""

      # List existing files in volume using Databricks Files REST API
      echo "Analyzing existing files in volume..."
      echo "Volume path: $VOLUME_PATH"

      TEMP_FILE="/tmp/existing_$$.txt"
      > "$TEMP_FILE"

      # Use Databricks Files REST API to list UC volume contents
      LIST_RESPONSE=$(curl -s -X GET \
        "${DATABRICKS_HOST}/api/2.0/fs/directories${VOLUME_PATH}" \
        -H "Authorization: Bearer $DATABRICKS_TOKEN" \
        -H "Content-Type: application/json")

      echo "List response status check..."
      # Parse filenames and sizes using jq (avoids YAML single-quote conflicts)
      echo "$LIST_RESPONSE" | jq -r \
        '.contents[]? | select(.path | endswith(".whl")) | ((.path | split("/") | last) + "|" + (.file_size | tostring))' \
        > "$TEMP_FILE" 2>/dev/null || true

      EXISTING_COUNT=$(wc -l < "$TEMP_FILE" 2>/dev/null | tr -d ' ')
      echo "Found $EXISTING_COUNT existing file(s) in volume"
      echo ""

      # Upload with smart skip
      UPLOAD_COUNT=0
      SKIP_COUNT=0
      OVERWRITE_COUNT=0
      FAIL_COUNT=0
      FILE_NUM=0

      for file in "$WHEELS_DIR"/*.whl; do
        FILE_NUM=$((FILE_NUM + 1))
        FILENAME=$(basename "$file")

        # Get local file size
        if [ "$(uname)" = "Darwin" ]; then
          LOCAL_SIZE=$(stat -f%z "$file" 2>/dev/null)
        else
          LOCAL_SIZE=$(stat -c%s "$file" 2>/dev/null)
        fi

        # Get remote file size from temp file
        REMOTE_SIZE=$(grep "^$FILENAME|" "$TEMP_FILE" 2>/dev/null | cut -d'|' -f2)

        # Decide action based on size comparison
        if [ -n "$REMOTE_SIZE" ] && [ "$LOCAL_SIZE" = "$REMOTE_SIZE" ]; then
          echo "[$FILE_NUM/$WHEEL_COUNT] $FILENAME - SKIP (identical, $LOCAL_SIZE bytes)"
          SKIP_COUNT=$((SKIP_COUNT + 1))
          continue
        elif [ -n "$REMOTE_SIZE" ]; then
          echo "[$FILE_NUM/$WHEEL_COUNT] $FILENAME - OVERWRITE (local: $LOCAL_SIZE bytes, remote: $REMOTE_SIZE bytes)"
          ACTION="overwrite"
        else
          echo "[$FILE_NUM/$WHEEL_COUNT] $FILENAME - NEW FILE ($LOCAL_SIZE bytes)"
          ACTION="new"
        fi

        # Upload using Databricks Files REST API (PUT /api/2.0/fs/files)
        TARGET_FILE_PATH="${VOLUME_PATH}/${FILENAME}"
        echo "  Source: $file"
        echo "  Target: $TARGET_FILE_PATH"

        set +e
        UPLOAD_OUTPUT=$(curl -s -w "\n%{http_code}" -X PUT \
          "${DATABRICKS_HOST}/api/2.0/fs/files${TARGET_FILE_PATH}?overwrite=true" \
          -H "Authorization: Bearer $DATABRICKS_TOKEN" \
          -H "Content-Type: application/octet-stream" \
          --data-binary "@$file" 2>&1)
        HTTP_CODE=$(echo "$UPLOAD_OUTPUT" | tail -1)
        RESPONSE_BODY=$(echo "$UPLOAD_OUTPUT" | head -n -1)
        set -e

        if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "201" ] || [ "$HTTP_CODE" = "204" ]; then
          echo "  Result: SUCCESS (HTTP $HTTP_CODE)"
          UPLOAD_COUNT=$((UPLOAD_COUNT + 1))
          [ "$ACTION" = "overwrite" ] && OVERWRITE_COUNT=$((OVERWRITE_COUNT + 1))
        else
          echo "  Result: FAILED (HTTP $HTTP_CODE)"
          echo "  $RESPONSE_BODY"
          FAIL_COUNT=$((FAIL_COUNT + 1))
        fi
        echo ""
      done

      rm -f "$TEMP_FILE"

      echo ""
      echo "=========================================="
      echo "Upload Summary"
      echo "=========================================="
      echo "Total files:         $WHEEL_COUNT"
      echo "Uploaded:            $UPLOAD_COUNT"
      echo "  - New files:       $((UPLOAD_COUNT - OVERWRITE_COUNT))"
      echo "  - Overwritten:     $OVERWRITE_COUNT"
      echo "Skipped (identical): $SKIP_COUNT"
      echo "Failed:              $FAIL_COUNT"
      echo "=========================================="
      echo ""

      if [ $FAIL_COUNT -gt 0 ]; then
        echo "ERROR: $FAIL_COUNT file(s) failed to upload!"
        exit 1
      fi

      if [ $SKIP_COUNT -eq $WHEEL_COUNT ]; then
        echo "All files already up-to-date - no upload needed!"
        echo "Time saved: ~90%"
      elif [ $SKIP_COUNT -gt 0 ]; then
        PERCENT=$((SKIP_COUNT * 100 / WHEEL_COUNT))
        echo "Upload completed with $SKIP_COUNT file(s) skipped!"
        echo "Time saved: ~${PERCENT}%"
      else
        echo "All files uploaded successfully!"
      fi

  env:
    DATABRICKS_HOST: ${{ parameters.DATABRICKS_HOST }}
    TARGET_ENV: ${{ parameters.TARGET_ENV }}
